In [22]:
# Funding
import pandas as pd
import numpy as np

def funding_report(symbol, side, volume_usd):
    # File
    file = f'/_data/base/{symbol}/csv/funding.csv'

    # Load funding
    df = pd.read_csv(file)
    df = df.sort_values('timestamp_ms', ignore_index=True)

    # Period detection
    period_ms = int(df['timestamp_ms'].diff().dropna().median())
    year_ms = 365 * 24 * 60 * 60 * 1000
    periods_per_year = year_ms / period_ms

    # Apply side per row
    rates = df['funding_rate'].to_numpy(dtype=np.float64)

    if side == 0:
        side_rates = np.zeros_like(rates)
    elif side == 1:
        # Long: pays when rate > 0, receives when rate < 0
        side_rates = -rates
    else:
        # Short: receives when rate > 0, pays when rate < 0
        side_rates = rates

    # Annualized rate (side-applied)
    rate_year = float(side_rates.mean()) * periods_per_year
    
    # Funding in USD per year (side-applied)
    funding_usd_year = 0.0 if side == 0 else rate_year * volume_usd
    
    return rate_year, funding_usd_year

In [ ]:
# Report
import os
from itables import show
from params import VOLUMES

csv = []

for symbol in {'BTCUSDC', 'ETHUSDC', 'SOLUSDC'}:
    for volume_usd in VOLUMES:
        for side in {-1, 1}:
            rate_year, funding_usd_year = funding_report(symbol, side, volume_usd)
            csv.append([symbol, volume_usd, side, rate_year, funding_usd_year])

df = pd.DataFrame(csv, columns=['symbol', 'volume_usd', 'side', 'rate_year', 'funding_usd_year'])
df.to_csv('funding.csv', index=False)

os.makedirs('reports', exist_ok=True)
df.to_csv('reports/funding_report.csv', index=False)

show(df, paging=False)


Loading ITables v2.6.2 from the internet... (need help?)
